## Preprocesamiento y Modelado

Una vez inspeccionado el dataset en `customer_churn_eda.ipynb` definimos una una estrategia de preprocesamiento iterativo (de menos a más) para el encontrar

In [347]:
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer,KNNImputer
from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score, f1_score, make_scorer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.discriminant_analysis import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB



### Configuración de constantes, rutas y variables 

En esta sección definimos constantes, rutas de archivos y atributos del dataset

In [348]:
# Rutas de los archivos de datos
TRAIN_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/train.csv"
TEST_PATH = "/kaggle/input/retencion-de-clientes-de-una-entidad-financiera/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# Cargamos los datos
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
# Definir variables objetivo
TARGET = 'Exited'
# Variables numéricas: Incluyo las continuas y las binarias numéricas (HasCrCard, IsActiveMember)
NUM_FEATURES = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
# Variables categóricas: Las de texto con pocas categorías
CAT_FEATURES = ['Geography', 'Gender', 'Surname']
# Variables a eliminar inicialmente (IDs y apellido)
DROP_FEATURES = ['CustomerId']
SURNAME_COL = 'Surname'
RANDOM_STATE = 100            # Semilla para reproducibilidad

### 1. Preparación de Datos

Separamos variables independientes y dependientes en X_train e y_train por convención.

In [349]:
# X_train = variables independientes
# y_train = variable dependiente u objetivo
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

## Clase para añadir features 

Creamos una clase para crear features sin fuga de datos. Esta clase aprende estadísticos (medianas y frecuencias) durante el fold de entrenamiento (`fit()`) y crea columnas nuevas usando esos estadístico, sin mirar la variable objetivo y sin fuga de datos tranformación(`tranform()`). No imputa ni escala de forma definitiva.

Activamos por familias.



In [350]:
# ---------- Feature Engineering ----------
from sklearn.base import BaseEstimator, TransformerMixin
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Genera features nuevas de forma segura (sin fuga):
    aprende mediana/frecuencias en fit() y las usa en transform().
    Activación por familias para ir de menos a más.
    """
    def __init__(
        self,
        # cada flag activa un grupo de features nuevas para probar mejoras
        # de forma incremental
        add_missing_count=True,     # cuenta de missings
        add_balance=True,           # añade features de Balance/Salary/ratios
        add_products=True,           # añade features de NumOfProducts
        add_age=True,                # añade features de Age/grupos de edad
        add_age_bins=True,           # añade variable categórica por tramos de edad
        add_interactions=True,       # añade features de interacciones simples
        add_surname_features=False   # añade features basadas en Surname
    ):
        self.add_missing_count = add_missing_count
        self.add_balance = add_balance
        self.add_products = add_products
        self.add_age = add_age
        self.add_age_bins = add_age_bins
        self.add_interactions = add_interactions
        self.add_surname_features = add_surname_features
   
    def fit(self, X, y=None):
        X = X.copy()
        # Calculamos medianas de columnas clave
        # lo hacemos en fit porque en validación cruzada, cada fold tiene un “train interno”
        # Si calculamos las medianas con todo el dataset,estaríamos usando info del fold de validación 
        # (fuga de datos) hacia el train
        # La hacemos en fit() para garantizar que cada fold de CV aprende sus propias medianas
        # Usamos medianas para evitar outliers y crear features sin NaN
        self.balance_median_ = X["Balance"].median(skipna=True)
        self.salary_median_ = X["EstimatedSalary"].median(skipna=True)
        self.creditscore_median_ = X["CreditScore"].median(skipna=True)
        self.age_median_ = X["Age"].median(skipna=True)
        # para la variable NumOfProducts (discreta) usamos moda (valor más frecuente)
        self.numprod_mode_ = X["NumOfProducts"].mode(dropna=True).iloc[0]

        # Calculamos frecuencia de apellidos para hacer frequency encoding (opcional)
        # tiene riesgo si en test hay apellidos no vistos en train
        # lo hacemos en fit() para evitar fuga de datos
        # Evitaría usar one-hot encoding de Surname por 
        # el alto cardinalidad (enorme número de categorías si cada cliente tiene un apellido distinto)
        # a veces mete ruido si hay muchos apellidos únicos
        if self.add_surname_features and "Surname" in X.columns:
            s = X["Surname"].astype("object")
            self.surname_freq_ = s.value_counts(dropna=True)
        else:
            self.surname_freq_ = None

        return self

    def transform(self, X):
        # Copia para retornarlo con las nuevas features añadidas
        X = X.copy()
        
        # --- Missing count por fila ---
        # add_indicator ya añade columnas binarias (crea una feature) por cada columna con missings
        # pero aquí añadimos una feature con el conteo total de missings por fila
        # La falta de datos indica el perfil del cliente,
        # un cliente con muchos datos faltantes puede ser menos comprometido, menos activo, menos interesado, etc.
        if self.add_missing_count:
            # columnas a considerar para el conteo de missings
            miss_cols = ["CreditScore","Balance","NumOfProducts","EstimatedSalary","HasCrCard"]
            # solo las que existan
            miss_cols = [c for c in miss_cols if c in X.columns]
            X["missing_count"] = X[miss_cols].isna().sum(axis=1)

        # --- Balance / Salary / ratios ---
        # features basadas en Balance y EstimatedSalary
        # Creamos "señales" de tipo "tengo saldo o no", "saldo cero", etc.
        # Creamos features logarítmicas para reducir el impacto de outliers y colas largas
        # los logaritmos ayudan a modelos lineales a capturar relaciones no lineales
        # La relación entre Balance y Salary puede indicar el nivel de ahorro o gasto del cliente
        # Creamos ratios entre Balance y Salary y su logaritmo para capturar la relación entre ambos
        # Si un cliente tiene un balance alto en comparación con su salario, puede indicar una mayor estabilidad financiera
        # o capacidad de ahorro, lo cual puede influir en su probabilidad de abandono
        if self.add_balance:
            balance_raw = bal_raw = pd.to_numeric(X["Balance"], errors="coerce")   # strings -> NaN
            balance = balance_raw.fillna(self.balance_median_)
            balance_0 = bal_raw.fillna(0)
            salary = X["EstimatedSalary"].fillna(self.salary_median_)
            # features binarias
            X["HasBalance"] = (balance_0 > 0).astype(int)     # 1 si Balance > 0 (NaN -> 0)
            X["Balance_is_zero"] = (balance_0 == 0).astype(int) # 1 si Balance == 0 (NaN -> 0).
            # features logarítmicas
            X["LogBalance"] = np.log1p(np.clip(balance, 0, None))
            X["LogSalary"] = np.log1p(np.clip(salary, 0, None))
            # ratios Balance/Salary
            X["BalToSal"] = balance / (salary + 1.0) # evitar división por cero
            X["LogBalToSal"] = np.log1p(np.clip(X["BalToSal"], 0, None)) # log(1 + ratio)
        
        # --- Número de productos (saltos típicos) ---
        # features basadas en NumOfProducts
        # NumOfProducts es una variable NO lineal (1->2->3) y va por saltos
        # así que creamos features binarias para cada salto típico
        # 1 producto, 2 productos, 3 o más productos
        if self.add_products:
            num_products = pd.to_numeric(X["NumOfProducts"], errors="coerce")
            num_products = num_products.fillna(self.numprod_mode_)
            X["Products_eq1"] = (num_products == 1).astype(int)
            X["Products_eq2"] = (num_products == 2).astype(int)
            X["Products_ge3"] = (num_products >= 3).astype(int)

        # --- Edad ---
        # features basadas en Age
        # Age tiene una relación no lineal con la retención de clientes
        # así que creamos Age al cuadrado para capturar esa no linealidad
        if self.add_age:
            X["Age2"] = X["Age"] ** 2
        
        #--- Age por tramos (bins) (opcional) ---
        # crea una variable categórica por tramos de edad
        # La edad puede influir en el comportamiento del cliente y su probabilidad de abandono
        # Creamos grupos de edad para capturar patrones específicos en diferentes rangos de edad
        # útil para modelos no lineales o árboles
        # puede meter ruido
        if self.add_age_bins:
            X["AgeBin"] = pd.cut(
                X["Age"],
                bins=[17, 25, 35, 45, 55, 100],
                labels=["18-25", "26-35", "36-45", "46-55", "56+"],
                include_lowest=True
            ).astype("object")
        
        # --- Interacciones simples ---
        # crea features de interacciones simples entre variables clave
        # interacciones típicas: 
        # Usuario inactivo si IsActiveMember == 0
        # Usuario inactivo y que tenga más de 3 productos (puede indicar un cliente 
        # con muchos productos pero poco comprometido)
        # País (geografía) y género para capturar si el efecto de género varía por país o
        # como indica la Female tiene más abandono en general
        if self.add_interactions:
            X["Inactive"] = (X["IsActiveMember"] == 0).astype(int)

            # interacción muy común: inactivo + 3+ productos
            # Los clientes con
            if "Products_ge3" in X.columns:
                X["Inactive_ge3prod"] = ((X["Inactive"] == 1) & (X["Products_ge3"] == 1)).astype(int)
            

            # interacción Geografía/género (Germany/Female)
            # crea variables binarias para cada combinación
            # one-hot encoding manual
            # Los alemanes son lo que más abandonan
            X["Germany"] = (X["Geography"] == "Germany").fillna(False).astype(int) # 1 si Germany 0 si no
            #X["France"] = (X["Geography"] == "France").fillna(False).astype(int)
            #X["Spain"] = (X["Geography"] == "Spain").fillna(False).astype(int)
            # Género female?
            X["Female"] = (X["Gender"] == "Female").fillna(False).astype(int)
            # interacciones female/pais
            X["Female_Germany"] = ((X["Female"] == 1) & (X["Germany"] == 1)).astype(int)
            #X["Female_France"] = ((X["Female"] == 1) & (X["France"] == 1)).astype(int)
            #X["Female_Spain"] = ((X["Female"] == 1) & (X["Spain"] == 1)).astype(int)
        
         # --- Surname ---
        # features basadas en Surname (opcional y más riesgoso)
        # añade features sobre Surname: missing, longitud y frecuencia
        # Frecuncia del apellido para 
        # tiene riesgo si en test hay apellidos no vistos en train
        if self.add_surname_features and "Surname" in X.columns:
            surname = X["Surname"].astype("object") # 
            X["Surname_missing"] = surname.isna().astype(int)   # 1 si Surname es NaN
            X["Surname_len"] = surname.fillna("").astype(str).str.len().astype(int) # longitud del apellido 0 si es NaN len(surname)
            # frecuencia del apellido (frequency encoding)
            if self.surname_freq_ is not None:
                X["Surname_freq"] = surname.map(self.surname_freq_).fillna(0).astype(float)
            else:
                X["Surname_freq"] = 0.0
        
        return X  # retornamos el DataFrame con las nuevas features añadidas

---------------------------------------

### Funciones auxiliar
#### Construir preprocesadores básico

La siguiente función genera un preprocesador básico que constituye la línea base de la que partimos en versiones anteriores.

In [351]:

def make_basic_preprocessor(X: pd.DataFrame):
    """Genera un ColumnTransformer con pipelines de preprocesamiento
    para variables numéricas y categóricas según la versión indicada.

    Args:
        X (pd.DataFrame): _input data frame_
        version (str): Versión del preprocesamiento en formato 'N#_C#'
        e.g. 'N3_C1' donde N# indica la versión numérica y C# la categórica
        numerical_cols (_type_): columnas numéricas para el preprocesamiento
        categorical_cols (_type_): columnas categóricas para el preprocesamiento

    Raises:
        ValueError: _unknown numeric version_
        ValueError: _unknown categorical version_

    Returns:
        _type_: ColumnTransformer con pipelines de preprocesamiento
    """
    # Base Line 
        # Imputamos todas las variables numéricas igual
        # Imputa valores faltantes con mediana + indicador de faltantes + escalado estandar
    numerical_cols = ['CreditScore', 'Age', 'Tenure', 'Balance', 
                          'NumOfProducts', 'EstimatedSalary', 'HasCrCard', 'IsActiveMember']
    categorical_cols = ['Geography', 'Gender']
    num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])
    # --- Categorical pipeline (si aplica) ---
    # Categóricas normales: imputar + onehot
    categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ])
   
    # Construimos el ColumnTransformer final
    # que une pipelines numéricos + pipelines categóricos
    col_trans_preprocessor = ColumnTransformer(
        transformers=[("num", num_pipe, numerical_cols), 
                      ("cat", categorical_pipe, categorical_cols)], 
        remainder="drop", # elimina columnas no especificadas
        verbose_feature_names_out=True) # nombres detallados de columnas
    return col_trans_preprocessor


#### Construir el preprocesador

In [352]:
# ---------- Preprocessor builder ----------
from sklearn.preprocessing import PowerTransformer


def make_preprocessor(use_surname: bool = False):
    # Columnas base
    numerical_continuous_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary"]
    numerical_discrete_cols = ["Tenure", "NumOfProducts"]
    numerical_binary_cols = ["HasCrCard", "IsActiveMember"]

    # --- Columnas features por grupo ---
    continuous_fe = []  # continuas creadas 
    discrete_fe = []  # discretas creadas
    binary_fe  = []  # binarias creadas

    # features que pueden existir según flags de FeatureEngineer
    discrete_fe += ["missing_count"]
    continuous_fe += ["LogBalance", "LogSalary", "BalToSal", "LogBalToSal", "Age2"]
    binary_fe  += ["HasBalance", "Balance_is_zero", 
                   "Products_eq1", "Products_eq2", "Products_ge3",
                   "Inactive", "Inactive_ge3prod", 
                   "Germany","Female", #"France","Spain"
                   "Female_Germany"]#,"Female_France","Female_Spain"]
    
    # Surname features (opcional)
     # tiene riesgo si en test hay apellidos no vistos en train
     # añade features sobre Surname: missing, longitud y frecuencia
     # Evitaría usar one-hot encoding de Surname por 
     # el alto cardinalidad (enorme número de categorías si cada cliente tiene un apellido distinto)
     # a veces mete ruido si hay muchos apellidos únicos
     # lo activamos solo si use_surname == True
    if use_surname:
        discrete_fe += ["Surname_len"]
        binary_fe  += ["Surname_missing"]
        continuous_fe += ["Surname_freq"]

    # --- Pipelines numéricos ---
    continuous_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("power", PowerTransformer(method="yeo-johnson")),
        ("scaler", RobustScaler()),
    ])

    # Feature Eng continuas: mejor NO aplicar power otra vez 
    continuous_fe_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", RobustScaler()), # RobustScaler para features con outliers
    ])

    discrete_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ])

    binary_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent", add_indicator=True)),
    ])

    # --- Categóricas ---
    cat_cols = ["Geography", "Gender", "AgeBin"]
    
    # ONE HOT ENCODER para convertir categóricas en numericas
    one_hot_encoder = OneHotEncoder(
        handle_unknown="ignore",
    )
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ])

    transformers = [
        ("cont", continuous_pipe, numerical_continuous_cols),
        ("cont_eng", continuous_fe_pipe, continuous_fe),
        ("disc", discrete_pipe, numerical_discrete_cols + discrete_fe),
        ("bin", binary_pipe, numerical_binary_cols + binary_fe),
        ("cat", categorical_pipe, cat_cols),
    ]

    preprocesor = ColumnTransformer(
        transformers=transformers,  # transformers 
        remainder="drop",           # elimina columnas no especificadas
        verbose_feature_names_out=True, # nombres detallados de columnas
        sparse_threshold=0.0 
    )

    return preprocesor

#### Construir pipelines

In [353]:
# Creación y evaluación del pipeline
# Construcción del pipeline con feature engineer, preprocesador y modelo

def make_pipeline(feature_engineer:FeatureEngineer, preprocessor: ColumnTransformer, model=None):
    """Construye un Pipeline con el preprocesador y el modelo indicado.
    Args:
        feature_engineer (FeatureEngineer): Objeto FeatureEngineer para crear nuevas features
        preprocessor (ColumnTransformer): Preprocesador ColumnTransformer
        model (_type_, optional): Modelo de clasificación. Defaults to None.
    Returns:
        Pipeline: Pipeline con preprocesador y modelo, si no se indica modelo
        se usa LinearDiscriminantAnalysis por defecto.
    """
    if model is None:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
        #model = LogisticRegression(max_iter=10000,solver="saga", random_state=RANDOM_STATE,class_weight='balanced',n_jobs=-1)
    if feature_engineer is None:
        return Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", model),
        ])
    else:
        return Pipeline([
        ("feature_engineer", feature_engineer),
        ("preprocessor", preprocessor),
        ("classifier", model),
        ])




#### Evaluar pipelines

Esta función evalua el pipeline usando validación cruzada estratificada

In [354]:
def evaluate_pipeline(pipe: Pipeline, X: pd.DataFrame, y: pd.Series, n_splits=5):
    """ Evalúa el pipeline usando Validación Cruzada estratificada
    Args:
        pipe (Pipeline): Pipeline a evaluar.
        X (pd.DataFrame): Datos de entrada.
        y (pd.Series): Datos objetivo.
        n_splits (int, optional): número de divisiones para Stratified K-Fold. Por defecto es 5.
    Returns:
        dict: Diccionario con las métricas promedio y desviación estándar.
    """
    # Configuramos la Validación Local Cruzada  n_splits splits (divisiones)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    # Definimos las métricas que queremos extraer
    # f1, roc_auc, precision, recall, accuracy son strings estándar de sklearn.
    # Kappa requiere make_scorer.
    scoring_metrics = {
        "f1": "f1",
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        'kappa': make_scorer(cohen_kappa_score),
        'precision': 'precision',
        'recall': 'recall',
    }
    cv_results = cross_validate(pipe, X, y, cv=cv, scoring=scoring_metrics, n_jobs=-1)
    return {
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_std":  cv_results["test_f1"].std(),
        "auc_mean": cv_results["test_roc_auc"].mean(),
        "auc_std":  cv_results["test_roc_auc"].std(),
        "accuracy_mean": cv_results["test_accuracy"].mean(),
        "accuracy_std":  cv_results["test_accuracy"].std(),
        "kappa_mean": cv_results["test_kappa"].mean(),
        "kappa_std":  cv_results["test_kappa"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_std":  cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_std":  cv_results["test_recall"].std()
    }

------------------
## Evaluación de diferentes features engineer y modelos

Probamos las diferentes variables que vamos creando en la clase `FeatureEngineer`. 

In [ ]:
# Configuración de experimentos
# modificamos Feature Engineer a probar 
# Feature Engineer 
feature_engineer = FeatureEngineer(
    add_missing_count=True,
    add_balance=True,
    add_products=True,
    add_age=True,
    add_age_bins=True,
    add_interactions=True,
    add_surname_features=True
)

use_surname_preprocesor = True

# Modelos a probar
models = {
    # Modelos lineales
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"),
    # Arboles
    # No necesitan escalado de variables
    "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced",n_jobs=-1,),
    "RandomForest_bal_subsample": RandomForestClassifier(n_estimators=400,random_state=RANDOM_STATE, 
                                           class_weight="balanced_subsample",n_jobs=-1,),
    # Otros modelos
    "NaiveBayes": GaussianNB(),
    "RedesNeurales": MLPClassifier(hidden_layer_sizes=(50,30), max_iter=1000, random_state=RANDOM_STATE,
                                   activation='relu',solver='adam',early_stopping=True),
    "KNN_5": KNeighborsClassifier(n_neighbors=5, n_jobs=-1,),
    "KNN_5_distance": KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),

}



#### Preprocesador base



In [356]:
preprocesors_results = []
base_preprocesor = make_basic_preprocessor(train_df)
# pipeline sin feature engineer y modelo por defecto
pipe = make_pipeline(None,base_preprocesor,None) 
print("Evaluando preprocesador base")
cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
preprocesors_results.append({"version": "Base", **cv_metrics})
display(pd.DataFrame(preprocesors_results))

Evaluando preprocesador base


,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
0,Base,0.32825,0.017747,0.768164,0.008146,0.80725,0.005585,0.238353,0.018927,0.567959,0.036011,0.231288,0.01535


#### Añadimos feature engineer

Creamos nuevas variables con la clase `FeatureEngineer`

In [357]:
preprocesor = make_preprocessor()
pipe = make_pipeline(feature_engineer, preprocesor)
cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
preprocesors_results.append({"version": "v1", **cv_metrics})

results_df = pd.DataFrame(preprocesors_results).sort_values("f1_mean", ascending=False)
display(results_df)
best_preprocesor_version = results_df.iloc[0]["version"]
print("Mejor versión de preprocesador:", best_preprocesor_version)

,version,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
1,v1,0.533646,0.005364,0.832450,0.006287,0.846375,0.004355,0.447853,0.008637,0.700915,0.027170,0.431288,0.007413
0,Base,0.328250,0.017747,0.768164,0.008146,0.807250,0.005585,0.238353,0.018927,0.567959,0.036011,0.231288,0.015350


Mejor versión de preprocesador: v1


#### Búsqueda del mejor modelo

Evaluamos los modelos con el mejor preprocesador encontrado 

In [358]:
model_results = []
# evaluamos todos los modelos con el preprocesador
for model_name, model in models.items():
     # Construimos el preprocesador fijo con la mejor versión
    preprocesor = make_preprocessor()
    pipe = make_pipeline(feature_engineer, preprocesor, model) 
    cv_metrics = evaluate_pipeline(pipe, X_train, y_train , n_splits=5)
    model_results.append({"modelo": model_name, **cv_metrics})

results_df = pd.DataFrame(model_results).sort_values("f1_mean", ascending=False)
best_model_name = results_df.iloc[0]["modelo"]
print("---- Resultados de validación cruzada de modelos con preprocesador fijo:----")
display(results_df)
print("Mejor modelo:", best_model_name)



---- Resultados de validación cruzada de modelos con preprocesador fijo:----


,modelo,f1_mean,f1_std,auc_mean,auc_std,accuracy_mean,accuracy_std,kappa_mean,kappa_std,precision_mean,precision_std,recall_mean,recall_std
1,LogisticRegression,0.565211,0.011428,0.837041,0.007267,0.765125,0.012860,0.417421,0.018278,0.454448,0.015978,0.748466,0.015641
0,LinearDiscriminantAnalysis,0.541786,0.008131,0.834487,0.006826,0.848125,0.005214,0.456495,0.011948,0.704722,0.029709,0.440491,0.006886
3,RandomForest,0.538761,0.010023,0.834390,0.008447,0.848375,0.001016,0.454249,0.008680,0.708925,0.012329,0.434969,0.016846
6,RedesNeurales,0.537948,0.016184,0.842599,0.006740,0.849125,0.005486,0.454409,0.018282,0.716645,0.029783,0.431288,0.019459
4,RandomForest_bal_subsample,0.535030,0.013278,0.834390,0.008326,0.847875,0.001750,0.450633,0.012214,0.709509,0.013283,0.430061,0.020476
2,DecisionTree,0.466336,0.018177,0.664749,0.011242,0.783625,0.008904,0.330667,0.023784,0.469067,0.021879,0.463804,0.016278
5,NaiveBayes,0.465893,0.021605,0.803903,0.003191,0.824000,0.007485,0.368089,0.018115,0.616325,0.041963,0.378528,0.042927
7,KNN_5,0.418154,0.020922,0.751939,0.013844,0.815375,0.006067,0.318842,0.023396,0.584268,0.026391,0.325767,0.018548
8,KNN_5_distance,0.416720,0.019348,0.754115,0.015610,0.812875,0.005557,0.314863,0.021619,0.570995,0.023563,0.328221,0.017352


Mejor modelo: LogisticRegression


## Construcción del pipeline final para Kaggle

Una vez obtenido la mejor combinación de preprocesadores y el mejor modelo, construimos el pipeline final para kaggle con la mejor combinación de ambos y volvemos a ejecutar la evaluación y el entrenamiento para finalmente obtener la predicción y generar el fichero para kaggle. 

In [ ]:
# Construcción del pipeline final para Kaggle con el mejor preprocesador y modelo
# Construimos el preprocesador fijo con la mejor versión
preprocesor = make_preprocessor(use_surname=use_surname_preprocesor)
best_model = models[best_model_name]
# Pipeline Completo (Preprocesamiento + Modelo)
best_model_pipeline = Pipeline(steps=[
    ('feature_engineer', feature_engineer),
    ('preprocessor', preprocesor),
    ('classifier', best_model)
])
# Configuramos y ejecutamos la Validación Cruzada local
cv_metrics = evaluate_pipeline(best_model_pipeline, X_train, y_train, n_splits=5)
# Generación de Submission para Kaggle con el mejor modelo encontrado
# Re-entrenamos con TODOS los datos de train para la predicción final
best_model_pipeline.fit(X_train, y_train) 
test_predictions = best_model_pipeline.predict(test_df)

# Crear fichero de salida
submission_df = pd.DataFrame({
    'CustomerId': test_df['CustomerId'],
    'Exited': test_predictions
})
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Fichero '{SUBMISSION_PATH}' generado correctamente.")

print("\n---- Mejores Resultados y Validación Cruzada local -----")
print("Mejor modelo:", best_model_name)
print("Mejor versión de preprocesador:", best_preprocesor_version)
print(f"Mean F1-Score:  {cv_metrics['f1_mean']:.4f} (+/- Std {cv_metrics['f1_std']:.4f})")
print(f"Mean Accuracy:  {cv_metrics['accuracy_mean']:.4f} (+/- Std {cv_metrics['accuracy_std']:.4f})")
print(f"Mean Kappa:     {cv_metrics['kappa_mean']:.4f}")
print(f"Mean Precision: {cv_metrics['precision_mean']:.4f}")
print(f"Mean Recall:    {cv_metrics['recall_mean']:.4f}")



Fichero '/kaggle/working/submission.csv' generado correctamente.

---- Mejores Resultados y Validación Cruzada local -----
Mejor modelo: LogisticRegression
Mejor versión de preprocesador: v1
Mean F1-Score:  0.5652 (+/- Std 0.0114)
Mean Accuracy:  0.7651 (+/- Std 0.0129)
Mean Kappa:     0.4174
Mean Precision: 0.4544
Mean Recall:    0.7485
